# Trade & Price: UN Comtrade Crude Oil Trade Data (HS 2709), 2019-2025

## Deliverable 1 – Task 1:

This notebook retrieves annual crude oil trade data from UN Comtrade for 2019-2025, including both exporter-reported (X) and importer-reported (M) trade flows.

Data source: [UN Comtrade API](https://comtradeapi.un.org/)

The official documentation for comtradeapicall is available at: [comtradeapicall](https://pypi.org/project/comtradeapicall/)

### A. API Setup

Import the required Python packages, load the API subscription key from .env, and define common query parameters for crude oil (HS 2709).

In [3]:
# Import standard python libraries
import pandas as pd
import numpy as np

In [4]:
# Run initially to install package from PyPi
# %pip install comtradeapicall

# Import comtradeapicall package
import comtradeapicall

# Get subscription key from .env
import os
from dotenv import load_dotenv

load_dotenv()

subscription_key = os.getenv("SUBSCRIPTION_KEY")

### B. Retrieve and Review Comtrade Data
Two transport-mode:

- All modes: No transport-mode restriction (motCode=None).
- Sea only: Maritime transport records (motCode=2100).

Both datasets include importer- and exporter-reported flows. Preview the returned records and inspect the available fields.

In [5]:
# Preview the data
comtrade_preview = comtradeapicall.previewFinalData(typeCode='C', freqCode='M', clCode='HS', period='202205',
                                        reporterCode=None, cmdCode='2709', flowCode='X,M', partnerCode=None,
                                        partner2Code=None, customsCode=None, motCode=None, maxRecords=500, format_output='JSON',
                                        aggregateBy=None, breakdownMode='classic', includeDesc=True)

comtrade_preview.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,M,20220501,2022,5,202205,24,AGO,Angola,M,...,NaN,False,0.0,False,4597.550,4574.63,4597.550,0,False,True
1,C,M,20220501,2022,5,202205,24,AGO,Angola,M,...,NaN,False,0.0,False,4546.310,4546.31,4546.310,0,False,True
2,C,M,20220501,2022,5,202205,24,AGO,Angola,M,...,8.655,False,0.0,False,51.240,28.32,51.240,0,False,True
3,C,M,20220501,2022,5,202205,28,ATG,Antigua and Barbuda,M,...,200.000,False,0.0,False,2395.074,NaN,2395.074,0,False,True
4,C,M,20220501,2022,5,202205,28,ATG,Antigua and Barbuda,M,...,200.000,False,0.0,False,2395.074,NaN,2395.074,0,False,True


In [6]:
comtrade_preview.columns

Index(['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period',
       'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc',
       'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code',
       'partner2ISO', 'partner2Desc', 'classificationCode',
       'classificationSearchCode', 'isOriginalClassification', 'cmdCode',
       'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc',
       'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty',
       'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty',
       'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt',
       'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue',
       'legacyEstimationFlag', 'isReported', 'isAggregate'],
      dtype='str')

In [7]:
# Common params for Crude Oil commodities between 2019-2025
crude_oil_filters = dict(
    typeCode='C', # Commodities
    # freqCode='A', # Annual
    clCode='HS',
    # period='2019,2020,2021,2022,2023,2024,2025',
    reporterCode=None, # No restrictions applied
    cmdCode='2709', # Crude oil
    flowCode='X,M', # Both exports and imports
    partnerCode=None, # No restrictions applied
    partner2Code=None,
    customsCode=None,
    format_output='JSON',
    aggregateBy=None,
    breakdownMode='plus', 
    countOnly=None,
    includeDesc=True # Include country and commodity descriptions
)

In [8]:
# Dataset A: No mode-of-transport restriction
comtrade_all_annual = comtradeapicall.getFinalData(
    subscription_key,
    motCode=None, # Include all modes of transport
    freqCode='A',
    period='2019,2020,2021,2022,2023,2024,2025',
    **crude_oil_filters
)

comtrade_all_annual.columns

Index(['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period',
       'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc',
       'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code',
       'partner2ISO', 'partner2Desc', 'classificationCode',
       'classificationSearchCode', 'isOriginalClassification', 'cmdCode',
       'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc',
       'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty',
       'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty',
       'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt',
       'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue',
       'legacyEstimationFlag', 'isReported', 'isAggregate'],
      dtype='str')

In [9]:
# Dataset A: No mode-of-transport restriction
comtrade_all_monthly = comtradeapicall.getFinalData(
    subscription_key,
    motCode=None, # Include all modes of transport
    freqCode='M',
    period='201905,201906',
    **crude_oil_filters
)

comtrade_all_monthly.columns == comtrade_all_annual.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True])

In [10]:
# Importing monthly-level reports needs 84 API calls
# Uncomment these lines of code if monthly-level report is needed

# import time

# years = range(2019, 2026)
# months = range(1, 13)

# monthly_dfs = []

# for year in years:
#     for month in months:
#         comtrade_monthly = comtradeapicall.getFinalData(
#             subscription_key,
#             motCode=None,
#             freqCode='M',
#             period=f'{year}{month:02d}',
#             **crude_oil_filters
#         )

#         print(f"Downloaded {period}: {len(comtrade_monthly)} rows")
        
#         monthly_dfs.append(comtrade_monthly)

#         time.sleep(1.1)

# comtrade_all_monthly = pd.DataFrame()
# comtrade_all_monthly = pd.concat(monthly_dfs, ignore_index=True)
# comtrade_all_monthly.shape

In [11]:
# Dataset B: Maritime transport only
comtrade_sea_annual = comtradeapicall.getFinalData(
    subscription_key,
    motCode='2100', # Mot code for 'Sea',
    freqCode='A',
    period='2019,2020,2021,2022,2023,2024,2025',
    **crude_oil_filters
)

comtrade_sea_annual.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20190101,2019,52,2019,24,AGO,Angola,M,...,1768.18,False,0.0,False,13738.600,12594.77,13738.600,0,False,True
1,C,A,20190101,2019,52,2019,24,AGO,Angola,M,...,268.18,False,0.0,False,1288.780,1274.31,1288.780,0,False,True
2,C,A,20190101,2019,52,2019,24,AGO,Angola,M,...,1500.00,False,0.0,False,12449.820,11320.46,12449.820,0,False,True
3,C,A,20190101,2019,52,2019,28,ATG,Antigua and Barbuda,M,...,135.60,False,0.0,False,580.737,NaN,580.737,0,False,True
4,C,A,20190101,2019,52,2019,28,ATG,Antigua and Barbuda,M,...,135.60,False,0.0,False,580.737,NaN,580.737,0,False,True


### C. Compare Reporting Countries

Find unique reporting countries and their UN Comtrade country codes in each dataset.

The initial retrieval contains 168 all-mode reporters and 61 sea-mode reporters, indicating more limited country coverage for sea-specific reporting.

In [12]:
all_reporters = set(comtrade_all_annual['reporterDesc'])
sea_reporters = set(comtrade_sea_annual['reporterDesc'])

print("All reporters:", len(all_reporters))
print("Sea reporters:", len(sea_reporters))

All reporters: 168
Sea reporters: 61


In [13]:
annual_reporters_all = (comtrade_all_annual[['reporterCode', 'reporterDesc']].drop_duplicates())
annual_reporters_sea = (comtrade_all_annual[['reporterCode', 'reporterDesc']].drop_duplicates())

In [14]:
annual_reporters_all.head()

,reporterCode,reporterDesc
0,4,Afghanistan
5,20,Andorra
7,24,Angola
10,28,Antigua and Barbuda
12,31,Azerbaijan


In [15]:
annual_reporters_sea.head()

,reporterCode,reporterDesc
0,4,Afghanistan
5,20,Andorra
7,24,Angola
10,28,Antigua and Barbuda
12,31,Azerbaijan


In [16]:
# Reporters with sea only data
sea_only_bool = annual_reporters_sea['reporterCode'].isin(annual_reporters_all['reporterCode'])
annual_reporters_sea.loc[~sea_only_bool]

,reporterCode,reporterDesc


### D. Process Data (Convert Units for Weight)
Convert net weight from kilograms to estimated barrels using:

> barrels = kg ÷ 1,000 × 7.3.

In [17]:
columns = [
    'reporterCode',
    'reporterDesc',
    'partnerCode',
    'partnerDesc',
    'partner2Code',
    'flowCode',
    'flowDesc',
    'motCode',
    'motDesc',
    'period',
    'netWgt',
    'primaryValue'
]

comtrade_all_annual_original = comtrade_all_annual.copy()
comtrade_all_sea_original = comtrade_sea_annual.copy()

comtrade_all_annual = comtrade_all_annual[columns]
comtrade_sea_annual = comtrade_sea_annual[columns]

In [18]:
comtrade_sea_annual

,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,flowCode,flowDesc,motCode,motDesc,period,netWgt,primaryValue
0,24,Angola,0,World,0,M,Import,2100,Sea,2019,1.768180e+03,1.373860e+04
1,24,Angola,724,Spain,0,M,Import,2100,Sea,2019,2.681800e+02,1.288780e+03
2,24,Angola,826,United Kingdom,0,M,Import,2100,Sea,2019,1.500000e+03,1.244982e+04
3,28,Antigua and Barbuda,0,World,0,M,Import,2100,Sea,2019,1.356000e+02,5.807370e+02
4,28,Antigua and Barbuda,842,USA,0,M,Import,2100,Sea,2019,1.356000e+02,5.807370e+02
...,...,...,...,...,...,...,...,...,...,...,...,...
8538,724,Spain,380,Italy,380,X,Export,2100,Sea,2025,7.664662e+07,5.455383e+07
8539,724,Spain,0,World,380,X,Export,2100,Sea,2025,7.664662e+07,5.455383e+07
8540,724,Spain,380,Italy,380,X,Export,2100,Sea,2025,7.664662e+07,5.455383e+07
8541,826,United Kingdom,0,World,380,X,Export,2100,Sea,2025,3.510845e+08,1.760186e+08


In [19]:
comtrade_all_annual['barrels'] =  (comtrade_all_annual['netWgt'] / 1000) * 7.3
comtrade_sea_annual['barrels'] =  (comtrade_sea_annual['netWgt'] / 1000) * 7.3

### E. Note the Gaps: Missing Years, Missing Major Exporters, and Mirror Mismatches
1. **Sea-Mode Reporter Table Fix:** Re-extract `annual_reporters_sea` directly from `comtrade_sea_annual` and list reporters with and without sea-mode (`motCode=2100`) coverage.
2. **Missing Years:** Check which reporting countries are missing years in the 2019–2025 window.
3. **Missing Major Exporters:** Check export (`flowCode='X'`) coverage across top global crude oil exporters.
4. **Exporter vs. Importer Mirror Mismatches:** Compare bilateral exporter-reported volumes (`X`) against importer-reported volumes (`M`) on total country-to-country rows (`partnerCode != 0`, `partner2Code == 0`, `motCode == 0`, `customsCode == 'C00'`).

In [21]:
# 1. Fix sea-only reporter table from comtrade_sea_annual
annual_reporters_sea = (
    comtrade_sea_annual[['reporterCode', 'reporterDesc']]
    .drop_duplicates()
    .sort_values(['reporterDesc', 'reporterCode'])
    .reset_index(drop=True)
)
reporters_without_sea = annual_reporters_all[
    ~annual_reporters_all['reporterCode'].isin(annual_reporters_sea['reporterCode'])
].reset_index(drop=True)

print(f"Total all-mode reporters: {len(annual_reporters_all)}")
print(f"Total sea-mode (motCode=2100) reporters: {len(annual_reporters_sea)}")
print(f"All-mode reporters missing sea-only data: {len(reporters_without_sea)}\n")

# 2. Missing years per reporter (2019-2025)
expected_years = set(range(2019, 2026))
years_per_reporter = (
    comtrade_all_annual.assign(year=comtrade_all_annual['period'].astype(int))
    .groupby(['reporterCode', 'reporterDesc'])['year']
    .apply(set)
    .reset_index()
)
years_per_reporter['missing_years'] = years_per_reporter['year'].apply(
    lambda yrs: sorted(expected_years - yrs)
)
incomplete_reporters = years_per_reporter[years_per_reporter['missing_years'].map(len) > 0]
print(f"Reporters with all 7 years (2019-2025): {len(years_per_reporter) - len(incomplete_reporters)}")
print(f"Reporters missing at least 1 year: {len(incomplete_reporters)}")
display(incomplete_reporters[['reporterCode', 'reporterDesc', 'missing_years']].head(15))

# 3. Check coverage of major crude oil exporters (Flow = 'X')
major_exporters = {
    682: "Saudi Arabia", 643: "Russia", 364: "Iran", 368: "Iraq",
    784: "United Arab Emirates", 414: "Kuwait", 842: "USA", 124: "Canada",
    579: "Norway", 76: "Brazil", 566: "Nigeria", 24: "Angola",
    398: "Kazakhstan", 484: "Mexico", 862: "Venezuela",
}
exp_only = comtrade_all_annual[comtrade_all_annual['flowCode'] == 'X'].copy()
exp_only['year'] = exp_only['period'].astype(int)
exp_years = exp_only.groupby('reporterCode')['year'].apply(lambda s: sorted(set(s))).to_dict()

exporter_coverage_rows = []
for code, name in major_exporters.items():
    yrs = exp_years.get(code, [])
    missing = sorted(expected_years - set(yrs))
    exporter_coverage_rows.append({
        "reporterCode": code,
        "Exporter": name,
        "Years_Reported_X": yrs,
        "Missing_Years_X": missing if yrs else "ALL (Not Reported)",
    })
exporter_coverage_df = pd.DataFrame(exporter_coverage_rows)
print("\nMajor Crude Oil Exporters Coverage (Exporter-Reported, Flow='X'):")
display(exporter_coverage_df)

# 4a. Masked exports to 'Areas, nes' (partnerCode == 899)
nes_totals = comtrade_all_annual_original[
    (comtrade_all_annual_original['flowCode'] == 'X')
    & (comtrade_all_annual_original['partnerCode'] == 899)
    & (comtrade_all_annual_original['partner2Code'].fillna(0) == 0)
    & (comtrade_all_annual_original['motCode'].fillna(0) == 0)
    & (comtrade_all_annual_original['customsCode'].fillna('C00') == 'C00')
].copy()
nes_totals['barrels'] = (pd.to_numeric(nes_totals['netWgt'], errors='coerce') / 1000.0) * 7.3
print("\nTop Masked Export Destinations Reported as 'Areas, nes' (Code 899):")
display(
    nes_totals.groupby(['refYear', 'reporterCode', 'reporterDesc', 'partnerDesc'], as_index=False)['barrels']
    .sum()
    .sort_values('barrels', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# 4b. True Country-to-Country Bilateral Mirror Mismatches (excluding World=0 and special codes)
special_codes = {0, 837, 838, 839, 899}
bilateral_totals = comtrade_all_annual_original[
    (~comtrade_all_annual_original['partnerCode'].isin(special_codes))
    & (~comtrade_all_annual_original['reporterCode'].isin(special_codes))
    & (comtrade_all_annual_original['partner2Code'].fillna(0) == 0)
    & (comtrade_all_annual_original['motCode'].fillna(0) == 0)
    & (comtrade_all_annual_original['customsCode'].fillna('C00') == 'C00')
].copy()
bilateral_totals['barrels'] = (pd.to_numeric(bilateral_totals['netWgt'], errors='coerce') / 1000.0) * 7.3

exp_reported = (
    bilateral_totals[bilateral_totals['flowCode'] == 'X']
    .groupby(['refYear', 'reporterCode', 'reporterDesc', 'partnerCode', 'partnerDesc'], as_index=False)['barrels']
    .sum()
    .rename(columns={
        'reporterCode': 'exporterCode', 'reporterDesc': 'exporterDesc',
        'partnerCode': 'importerCode', 'partnerDesc': 'importerDesc',
        'barrels': 'exporter_bbl',
    })
)

imp_reported = (
    bilateral_totals[bilateral_totals['flowCode'] == 'M']
    .groupby(['refYear', 'partnerCode', 'partnerDesc', 'reporterCode', 'reporterDesc'], as_index=False)['barrels']
    .sum()
    .rename(columns={
        'partnerCode': 'exporterCode', 'partnerDesc': 'exporterDesc_imp',
        'reporterCode': 'importerCode', 'reporterDesc': 'importerDesc_imp',
        'barrels': 'importer_bbl',
    })
)

mirror_df = exp_reported.merge(
    imp_reported,
    on=['refYear', 'exporterCode', 'importerCode'],
    how='outer',
)
mirror_df['exporterDesc'] = mirror_df['exporterDesc'].fillna(mirror_df['exporterDesc_imp'])
mirror_df['importerDesc'] = mirror_df['importerDesc'].fillna(mirror_df['importerDesc_imp'])
mirror_df['exporter_bbl'] = mirror_df['exporter_bbl'].fillna(0.0)
mirror_df['importer_bbl'] = mirror_df['importer_bbl'].fillna(0.0)
mirror_df['abs_diff_bbl'] = (mirror_df['exporter_bbl'] - mirror_df['importer_bbl']).abs()

print("\nTop 15 Country-to-Country Bilateral Mirror Mismatches (Exporter X vs. Importer M, in barrels):")
display(
    mirror_df[['refYear', 'exporterDesc', 'importerDesc', 'exporter_bbl', 'importer_bbl', 'abs_diff_bbl']]
    .sort_values('abs_diff_bbl', ascending=False)
    .head(15)
    .reset_index(drop=True)
)

Total all-mode reporters: 168
Total sea-mode (motCode=2100) reporters: 61
All-mode reporters missing sea-only data: 107

Reporters with all 7 years (2019-2025): 89
Reporters missing at least 1 year: 79


,reporterCode,reporterDesc,missing_years
0,4,Afghanistan,"[2020, 2021, 2022, 2023, 2024, 2025]"
1,12,Algeria,"[2019, 2020, 2021, 2022, 2023, 2025]"
2,20,Andorra,[2024]
4,28,Antigua and Barbuda,"[2020, 2025]"
9,44,Bahamas,[2025]
10,48,Bahrain,"[2021, 2022, 2025]"
11,51,Armenia,"[2019, 2020, 2021, 2022, 2025]"
14,60,Bermuda,"[2019, 2020, 2021, 2023, 2024, 2025]"
15,64,Bhutan,"[2021, 2022, 2023, 2024, 2025]"
18,72,Botswana,[2025]



Major Crude Oil Exporters Coverage (Exporter-Reported, Flow='X'):


,reporterCode,Exporter,Years_Reported_X,Missing_Years_X
0,682,Saudi Arabia,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",[]
1,643,Russia,"[2019, 2020, 2021]","[2022, 2023, 2024, 2025]"
2,364,Iran,"[2019, 2020, 2021, 2022, 2023]","[2024, 2025]"
3,368,Iraq,"[2021, 2022, 2023, 2024]","[2019, 2020, 2025]"
4,784,United Arab Emirates,"[2019, 2020, 2021, 2022, 2023]","[2024, 2025]"
5,414,Kuwait,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",[]
6,842,USA,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",[]
7,124,Canada,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",[]
8,579,Norway,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",[]
9,76,Brazil,"[2019, 2020, 2021, 2022, 2023, 2024, 2025]",[]



Top Masked Export Destinations Reported as 'Areas, nes' (Code 899):


,refYear,reporterCode,reporterDesc,partnerDesc,barrels
0,2019,682,Saudi Arabia,"Areas, nes",2.601001e+09
1,2022,682,Saudi Arabia,"Areas, nes",2.558407e+09
2,2020,784,United Arab Emirates,"Areas, nes",2.530327e+09
3,2020,682,Saudi Arabia,"Areas, nes",2.446299e+09
4,2023,682,Saudi Arabia,"Areas, nes",2.364807e+09
5,2021,682,Saudi Arabia,"Areas, nes",2.258821e+09
6,2024,682,Saudi Arabia,"Areas, nes",2.194073e+09
7,2023,784,United Arab Emirates,"Areas, nes",2.186861e+09
8,2019,784,United Arab Emirates,"Areas, nes",1.983201e+09
9,2022,784,United Arab Emirates,"Areas, nes",1.855691e+09



Top 15 Country-to-Country Bilateral Mirror Mismatches (Exporter X vs. Importer M, in barrels):


,refYear,exporterDesc,importerDesc,exporter_bbl,importer_bbl,abs_diff_bbl
0,2025,Saudi Arabia,"Other Asia, nes",1.883889e+09,0.000000e+00,1.883889e+09
1,2023,United Arab Emirates,Thailand,0.000000e+00,1.568873e+09,1.568873e+09
2,2024,Russian Federation,China,0.000000e+00,7.918255e+08,7.918255e+08
3,2025,Russian Federation,China,0.000000e+00,7.362439e+08,7.362439e+08
4,2023,Russian Federation,China,0.000000e+00,7.267174e+08,7.267174e+08
5,2024,Russian Federation,India,0.000000e+00,6.716647e+08,6.716647e+08
6,2023,Russian Federation,India,0.000000e+00,6.491176e+08,6.491176e+08
7,2025,Russian Federation,India,0.000000e+00,6.476792e+08,6.476792e+08
8,2023,Saudi Arabia,China,0.000000e+00,6.452585e+08,6.452585e+08
9,2021,Saudi Arabia,China,0.000000e+00,6.391678e+08,6.391678e+08
